# Практична робота №6 — Транспортна задача та задача про призначення

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою матрицю тарифів, запаси та потреби автоматично.

In [ ]:
import numpy as np
import pandas as pd
import time
import itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import linprog, linear_sum_assignment

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Постановка задачі та перевірка балансу

Матриця C — тарифи перевезення від постачальника i до споживача j. Запаси a_i, потреби b_j. Задача закрита, якщо Σa_i = Σb_j — інакше потрібен фіктивний пункт із нульовими тарифами.

In [ ]:
C_raw, a_raw, b_raw = VARIANTS[VARIANT]
C = np.array(C_raw, dtype=float)
a = np.array(a_raw, dtype=float)
b = np.array(b_raw, dtype=float)
m, n = C.shape

print(f"Варіант {VARIANT}")
print("C ="); print(C)
print("a =", a_raw, " b =", b_raw)
print(f"Σa = {a.sum():.0f}, Σb = {b.sum():.0f} -> "
      f"{'ЗАКРИТА' if abs(a.sum()-b.sum())<1e-9 else 'ВІДКРИТА — потрібен фіктивний пункт'}")

## 2. Побудова опорного плану трьома методами

Північно-західний кут ігнорує тарифи. Мінімальний елемент і метод Фогеля враховують вартість.

In [ ]:
def cost(X):
    return float((np.array(X) * C).sum())


def north_west(a, b):
    a, b = a.copy(), b.copy()
    X = np.zeros((len(a), len(b)))
    i = j = 0
    while i < len(a) and j < len(b):
        q = min(a[i], b[j])
        X[i, j] = q
        a[i] -= q; b[j] -= q
        if a[i] <= 1e-9:
            i += 1
        else:
            j += 1
    return X


def min_element(C, a, b):
    a, b = a.copy(), b.copy()
    X = np.zeros_like(C); M = C.copy()
    while a.sum() > 1e-9:
        i, j = np.unravel_index(np.argmin(M), M.shape)
        q = min(a[i], b[j])
        X[i, j] = q
        a[i] -= q; b[j] -= q
        if a[i] <= 1e-9: M[i, :] = np.inf
        if b[j] <= 1e-9: M[:, j] = np.inf
    return X


def vogel(C, a, b):
    a, b = a.copy(), b.copy()
    M = C.copy()
    rows_open, cols_open = set(range(len(a))), set(range(len(b)))
    X = np.zeros_like(C)

    def penalty(vals):
        v = sorted(vals)
        return v[0] if len(v) == 1 else v[1] - v[0]

    while rows_open and cols_open:
        row_pen = {i: penalty([M[i, j] for j in cols_open]) for i in rows_open}
        col_pen = {j: penalty([M[i, j] for i in rows_open]) for j in cols_open}
        best_row = max(row_pen, key=row_pen.get)
        best_col = max(col_pen, key=col_pen.get)
        if row_pen[best_row] >= col_pen[best_col]:
            i = best_row; j = min(cols_open, key=lambda c: M[i, c])
        else:
            j = best_col; i = min(rows_open, key=lambda r: M[r, j])
        q = min(a[i], b[j])
        X[i, j] = q
        a[i] -= q; b[j] -= q
        if a[i] <= 1e-9: rows_open.discard(i)
        if b[j] <= 1e-9: cols_open.discard(j)
    return X


X_nw, X_me, X_vg = north_west(a, b), min_element(C, a, b), vogel(C, a, b)
for name, X in [("пн.-зх. кут", X_nw), ("мін. елемент", X_me), ("Фогель", X_vg)]:
    print(f"{name}: F = {cost(X):.0f}")
    print(X.astype(int))

## 3. Ручна ітерація методу потенціалів

Потенціали u_i, v_j знаходимо з u_i+v_j=c_ij для базисних клітинок плану мінімального елемента (u_1=0 довільно), далі обчислюємо оцінки вільних клітинок Δ_ij = c_ij-(u_i+v_j).

In [ ]:
def basic_cells(X, tol=1e-6):
    return [(i, j) for i in range(m) for j in range(n) if X[i, j] > tol]


def potentials_from_basis(C, basis, m, n):
    u = [None] * m; v = [None] * n
    adj = {}
    for (i, j) in basis:
        adj.setdefault(("r", i), []).append(("c", j, C[i, j]))
        adj.setdefault(("c", j), []).append(("r", i, C[i, j]))
    known = {("r", 0): 0.0}
    frontier = [("r", 0)]
    while frontier:
        node = frontier.pop()
        for kind, idx, c_ij in adj.get(node, []):
            other = (kind, idx)
            if other not in known:
                known[other] = c_ij - known[node]
                frontier.append(other)
    for i in range(m):
        if ("r", i) in known: u[i] = known[("r", i)]
    for j in range(n):
        if ("c", j) in known: v[j] = known[("c", j)]
    return u, v


basis_me = basic_cells(X_me)
u_manual, v_manual = potentials_from_basis(C, basis_me, m, n)
print("u =", u_manual)
print("v =", v_manual)

deltas = np.full((m, n), np.nan)
for i in range(m):
    for j in range(n):
        if (i, j) not in basis_me and u_manual[i] is not None and v_manual[j] is not None:
            deltas[i, j] = C[i, j] - (u_manual[i] + v_manual[j])
print("Оцінки вільних клітинок Δ:"); print(deltas)

## 4. Цикл перерахунку (якщо план ще не оптимальний)

Якщо є клітинка з Δ<0, будуємо для неї цикл перерахунку (замкнена ламана по базисних клітинках) і зсуваємо план на θ = min перевезень у «мінусових» вершинах циклу.

In [ ]:
def find_cycle(X, start, tol=1e-6):
    basis = set(basic_cells(X, tol)) | {start}
    rows, cols = {}, {}
    for (i, j) in basis:
        rows.setdefault(i, []).append(j)
        cols.setdefault(j, []).append(i)

    def dfs(path, visited, horizontal):
        last = path[-1]
        candidates = cols[last[1]] if horizontal else rows[last[0]]
        for nxt in candidates:
            cell = (nxt, last[1]) if horizontal else (last[0], nxt)
            if cell == start and len(path) >= 3:
                return path + [cell]
            if cell in visited or cell == path[-1]:
                continue
            res = dfs(path + [cell], visited | {cell}, not horizontal)
            if res:
                return res
        return None

    return dfs([start], {start}, True)


free_cells = [(i, j) for i in range(m) for j in range(n)
              if (i, j) not in basis_me and not np.isnan(deltas[i, j])]
worst_cell = min(free_cells, key=lambda ij: deltas[ij]) if free_cells else None

if worst_cell and deltas[worst_cell] < -1e-9:
    cycle = find_cycle(X_me, worst_cell)[:-1]
    minus_cells = cycle[1::2]
    theta = min(X_me[i, j] for (i, j) in minus_cells)
    X_me_improved = X_me.copy()
    for idx, (i, j) in enumerate(cycle):
        X_me_improved[i, j] += theta if idx % 2 == 0 else -theta
    print("Цикл:", cycle, " θ =", theta)
    print("Новий план:"); print(X_me_improved.astype(int))
    print("F =", cost(X_me_improved))
else:
    print("План мінімального елемента вже оптимальний (усі Δ >= 0).")

## 5. Оптимальний план (LP, scipy) і потенціали як двоїсті змінні

In [ ]:
c_vec = C.flatten()
A_eq, b_eq = [], []
for i in range(m):
    row = np.zeros(m * n); row[i*n:(i+1)*n] = 1
    A_eq.append(row); b_eq.append(a_raw[i])
for j in range(n):
    row = np.zeros(m * n)
    for i in range(m): row[i*n+j] = 1
    A_eq.append(row); b_eq.append(b_raw[j])

res_lp = linprog(c_vec, A_eq=A_eq, b_eq=b_eq, bounds=(0, None), method="highs")
X_opt = res_lp.x.reshape(m, n)
u_lp, v_lp = res_lp.eqlin.marginals[:m], res_lp.eqlin.marginals[m:]

print("Оптимальний план:"); print(X_opt.round(1))
print(f"F* = {res_lp.fun:.0f}")
print("u =", np.round(u_lp, 2), " v =", np.round(v_lp, 2))

basis_opt = basic_cells(X_opt)
ok = all(abs(u_lp[i] + v_lp[j] - C[i, j]) < 1e-6 for (i, j) in basis_opt)
print("u_i+v_j=c_ij на базисних клітинках:", "ВИКОНУЄТЬСЯ" if ok else "є розбіжність")

## 6. Задача про призначення (угорський алгоритм)

Матрицю тарифів доповнюємо до квадратної фіктивним постачальником (нульові тарифи), якщо вона не квадратна, і застосовуємо `linear_sum_assignment`.

In [ ]:
size = max(m, n)
T = np.zeros((size, size))
T[:m, :n] = C

row_assign, col_assign = linear_sum_assignment(T)
print("Призначення (рядок -> стовпець):", {int(i)+1: int(j)+1 for i, j in zip(row_assign, col_assign)})
print("Сумарні витрати:", T[row_assign, col_assign].sum())

## 7. Трудомісткість: угорський алгоритм проти повного перебору

In [ ]:
rng = np.random.default_rng(42)
sizes = [4, 6, 8, 10]
brute_sizes = [4, 6, 8]
hu_times, brute_times = [], []
for size in sizes:
    M_rand = rng.integers(1, 50, size=(size, size)).astype(float)
    t0 = time.perf_counter()
    linear_sum_assignment(M_rand)
    hu_times.append(time.perf_counter() - t0)
    if size in brute_sizes:
        t0 = time.perf_counter()
        best = min(sum(M_rand[i, p[i]] for i in range(size)) for p in itertools.permutations(range(size)))
        brute_times.append(time.perf_counter() - t0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(sizes, [max(t, 1e-7) for t in hu_times], "o-", label="Угорський алгоритм")
ax.semilogy(brute_sizes, [max(t, 1e-7) for t in brute_times], "s-", label="Повний перебір")
ax.set_xlabel("n"); ax.set_ylabel("час, с (лог. шкала)"); ax.legend(); ax.grid(alpha=.3, which="both")
ax.set_title("Угорський алгоритм проти повного перебору")
fig.tight_layout()
fig.savefig("complexity.png", dpi=150)
fig

## 8. Зведена таблиця порівняння опорних планів і оптимуму

In [ ]:
pd.DataFrame([
    dict(method="Північно-західний кут", F=cost(X_nw)),
    dict(method="Мінімальний елемент", F=cost(X_me)),
    dict(method="Фогель", F=cost(X_vg)),
    dict(method="Оптимум (LP)", F=res_lp.fun),
])